# Ground-Truth-Based Threshold Calibration for `classify_habitat()`

Pure local reimplementation -- no `ee`/`eetools`/`geemap` imports. Compares `classify_habitat()`'s
numpy reimplementation against the Airbus 1m RF ground-truth layer, to iterate on
`config.DW_HABITAT_THRESHOLDS` in seconds instead of round-tripping through Earth Engine. See
`notes.md`'s "DW Historical Change Detection" section for the full calibration history and
results.

**Inputs** (grid-identical: EPSG:32736, 10m, 1346x1222):
- `outputs/rasters/dynamic_world/DW_connectivity_inputs_current_2022_2025_project.tif` -- the
  9-band derived probability stack.
- `outputs/rasters/dynamic_world/connectivity_inputs_DW_raw_crops_built_bare_current_2022_2025_project.tif`
  -- raw `crops`/`built`/`bare` bands (`classify_habitat()` needs these directly;
  `hard_conversion_prob = crops + built` can't be un-summed).
- `outputs/rf_hab_classifier/airbus_landcover_classification_10m_clipped.tif` -- the RF ground
  truth, crosswalked to the DW habitat scheme via `config.RF_TO_DW_HABITAT_CROSSWALK`.

**Accepted risk**: this is a second implementation of `classify_habitat()`'s rule logic, so it can
drift from the real Earth Engine version. Once thresholds are chosen here, the mandatory final
step is to update `config.DW_HABITAT_THRESHOLDS`, re-run the real EE `classify_habitat()` for the
current period only, and pixel-compare its output against this notebook's local classification
for the same thresholds (see the last section).


In [ ]:
import numpy as np
import pandas as pd
import rasterio
from sklearn.metrics import cohen_kappa_score, confusion_matrix

import config

pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:0.3f}")


### Load rasters into memory (one-time; ~1.6M pixels each, trivial for numpy)

In [ ]:
DW_STACK_PATH = config.DW_RASTER_INPUT_DIR / "DW_connectivity_inputs_current_2022_2025_project.tif"
DW_RAW_PATH = config.DW_RASTER_INPUT_DIR / "connectivity_inputs_DW_raw_crops_built_bare_current_2022_2025_project.tif"
RF_REFERENCE_PATH = config.RF_CLASSIFIER_DIR / "airbus_landcover_classification_10m_clipped.tif"


def read_named_bands(path):
    """Read a multi-band raster into a dict keyed by each band's `descriptions` entry."""
    with rasterio.open(path) as src:
        names = src.descriptions
        if any(n is None for n in names):
            raise ValueError(f"{path} is missing band descriptions: {names}")
        arr = src.read(masked=True)
        profile = src.profile
    return {name: arr[i] for i, name in enumerate(names)}, profile


dw_stack, dw_profile = read_named_bands(DW_STACK_PATH)
dw_raw, dw_raw_profile = read_named_bands(DW_RAW_PATH)

assert dw_profile["width"] == dw_raw_profile["width"] and dw_profile["height"] == dw_raw_profile["height"], (
    "DW derived stack and raw crops/built/bare export are not grid-identical -- re-check exports."
)

bands = {**dw_stack, **dw_raw}
print("Loaded bands:", sorted(bands.keys()))
print("Grid shape:", dw_profile["height"], "x", dw_profile["width"])


### `classify_habitat_np()` -- exact numpy mirror of the EE version

Mirrors `historical_change_detection.ipynb`'s `classify_habitat()` (cell `c14`) rule-for-rule,
including the sequential-overwrite precedence. Each EE `.where(cond, value)` call overwrites
*whatever the running image currently holds* wherever `cond` is true -- so the rule applied LAST
wins for any pixel matched by more than one rule. The real order is (lowest precedence first):
natural(3) -> grass(2) -> woody(1) -> bare(6) -> water(7) -> built(5) -> crops(4), making cropland
the highest-precedence class. This ordering is the single highest-risk drift point between the two
implementations -- **do not reorder these `arr[mask] = value` lines** without re-checking `c14`.


In [ ]:
def classify_habitat_np(bands, thresholds=None, min_obs=None):
    """Numpy reimplementation of classify_habitat(). Returns a uint8 array, 0 = masked/nodata."""
    t = {**config.DW_HABITAT_THRESHOLDS, **(thresholds or {})}
    min_obs = config.DW_MIN_OBS_ANNUAL if min_obs is None else min_obs

    natural = bands["natural_prob"]
    woody = bands["woody_prob"]
    grass = bands["grass_prob"]
    crops = bands["crops"]
    built = bands["built"]
    bare = bands["bare"]
    water_wetland = bands["water_wetland_prob"]
    top1 = bands["top1_prob"]
    valid_obs_count = bands["valid_obs_count"]

    habitat_class = np.full(natural.shape, 8, dtype=np.uint8)  # default: uncertain / low confidence

    habitat_class[natural.filled(-np.inf) >= t["natural_min"]] = 3
    habitat_class[
        (grass.filled(-np.inf) >= t["grass_min"]) & ((grass - woody).filled(-np.inf) >= t["woody_grass_margin"])
    ] = 2
    habitat_class[
        (woody.filled(-np.inf) >= t["woody_min"]) & ((woody - grass).filled(-np.inf) >= t["woody_grass_margin"])
    ] = 1
    habitat_class[
        (bare.filled(-np.inf) >= t["bare_min"]) & (bare.filled(-np.inf) > crops.filled(np.inf))
        & (bare.filled(-np.inf) > built.filled(np.inf))
    ] = 6
    habitat_class[water_wetland.filled(-np.inf) >= t["water_wetland_min"]] = 7
    habitat_class[built.filled(-np.inf) >= t["built_min"]] = 5
    habitat_class[crops.filled(-np.inf) >= t["crops_min"]] = 4

    valid = valid_obs_count.filled(-np.inf) >= min_obs
    valid &= top1.filled(-np.inf) >= t["top1_min"]
    for band in bands.values():
        valid &= ~np.ma.getmaskarray(band)

    out = np.where(valid, habitat_class, 0).astype(np.uint8)
    return out


baseline_class = classify_habitat_np(bands)
values, counts = np.unique(baseline_class, return_counts=True)
pd.Series(counts, index=[config.DW_HABITAT_CLASS_LABELS.get(v, "nodata(0)") for v in values], name="pixel_count")


### Load the RF ground-truth reference, crosswalked to the DW habitat scheme

In [ ]:
def load_rf_reference(path, crosswalk):
    """Read the RF classification and remap its class IDs onto the DW habitat scheme via a 256-entry LUT."""
    with rasterio.open(path) as src:
        rf_raw = src.read(1, masked=True)

    lut = np.zeros(256, dtype=np.uint8)  # unmapped IDs (incl. nodata) -> 0
    for rf_id, dw_id in crosswalk.items():
        lut[rf_id] = dw_id

    rf_filled = rf_raw.filled(0)
    rf_dw = lut[rf_filled]
    rf_dw[np.ma.getmaskarray(rf_raw)] = 0
    return rf_dw


rf_reference = load_rf_reference(RF_REFERENCE_PATH, config.RF_TO_DW_HABITAT_CROSSWALK)
values, counts = np.unique(rf_reference, return_counts=True)
pd.Series(counts, index=[config.DW_HABITAT_CLASS_LABELS.get(v, "nodata(0)") for v in values], name="pixel_count")


### `compare_to_reference()` -- confusion matrix + accuracy metrics

Only pixels with a valid RF reference class (nonzero) AND a valid DW prediction (nonzero) are
scored -- pixels the RF classifier itself couldn't confidently label are not usable ground truth,
and DW pixels masked by `valid_obs_count`/`top1_prob` have no classification to score. DW's
Mixed natural(3)/Uncertain(8) classes never appear in `rf_reference` by construction (no RF class
crosswalks to them, see `config.RF_TO_DW_HABITAT_CROSSWALK`), so their producer's accuracy is
reported as `NaN`, not a misleading `0.0`.


In [ ]:
DW_CLASS_IDS = config.DW_HABITAT_CLASS_CODES  # 1..8
DW_CLASS_NAMES = [config.DW_HABITAT_CLASS_LABELS[i] for i in DW_CLASS_IDS]


def compare_to_reference(dw_pred, rf_ref, class_ids=DW_CLASS_IDS, class_names=DW_CLASS_NAMES):
    scoreable = (dw_pred != 0) & (rf_ref != 0)
    y_true = rf_ref[scoreable]
    y_pred = dw_pred[scoreable]

    cm = confusion_matrix(y_true, y_pred, labels=class_ids)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

    overall_accuracy = np.trace(cm) / cm.sum() if cm.sum() else float("nan")
    kappa = cohen_kappa_score(y_true, y_pred, labels=class_ids) if cm.sum() else float("nan")

    row_sums = cm.sum(axis=1)  # true-class totals -- 0 for classes never present in rf_ref (3, 8)
    col_sums = cm.sum(axis=0)  # predicted-class totals
    diag = np.diag(cm)
    with np.errstate(invalid="ignore", divide="ignore"):
        producers = np.where(row_sums > 0, diag / row_sums, np.nan)  # a.k.a. recall
        users = np.where(col_sums > 0, diag / col_sums, np.nan)  # a.k.a. precision

    per_class = pd.DataFrame(
        {
            "n_reference_pixels": row_sums,
            "n_predicted_pixels": col_sums,
            "producers_accuracy": producers,
            "users_accuracy": users,
        },
        index=class_names,
    )

    return {
        "confusion_matrix": cm_df,
        "overall_accuracy": overall_accuracy,
        "kappa": kappa,
        "per_class": per_class,
        "n_scored_pixels": int(scoreable.sum()),
        "n_unscoreable_pixels": int((~scoreable).sum()),
    }


def print_comparison(result, label=""):
    if label:
        print(f"=== {label} ===")
    print(f"Scored pixels: {result['n_scored_pixels']:,} (unscoreable: {result['n_unscoreable_pixels']:,})")
    print(f"Overall accuracy: {result['overall_accuracy']:.4f}   Kappa: {result['kappa']:.4f}")
    print()
    print("Confusion matrix (rows = RF reference, columns = DW prediction):")
    print(result["confusion_matrix"])
    print()
    print("Per-class accuracy:")
    print(result["per_class"])


### Baseline: current, unmodified `config.DW_HABITAT_THRESHOLDS` -- the reference point for tuning

In [ ]:
baseline_result = compare_to_reference(baseline_class, rf_reference)
print_comparison(baseline_result, label="Baseline (unmodified config.DW_HABITAT_THRESHOLDS)")


### Iterative calibration harness

`evaluate_thresholds()` loads nothing new -- both rasters are already in memory, so each call is a
sub-second, pure-numpy reclassification. `sweep_threshold()` is a convenience one-parameter
sensitivity sweep, holding every other threshold at its `config.DW_HABITAT_THRESHOLDS` default.

This is a manual, inspect-then-adjust loop, not a blind grid search -- 9 interacting thresholds
makes brute-force search impractical, and the confusion matrix tells you *which* class is
misclassified as *what*, which a single accuracy number does not.


In [ ]:
def evaluate_thresholds(overrides=None, min_obs=None, label=""):
    dw_pred = classify_habitat_np(bands, thresholds=overrides, min_obs=min_obs)
    result = compare_to_reference(dw_pred, rf_reference)
    if label:
        print_comparison(result, label=label)
    return result


def sweep_threshold(param_name, values, min_obs=None):
    rows = []
    for v in values:
        result = evaluate_thresholds(overrides={param_name: v}, min_obs=min_obs)
        rows.append(
            {
                param_name: v,
                "overall_accuracy": result["overall_accuracy"],
                "kappa": result["kappa"],
            }
        )
    return pd.DataFrame(rows)


### Automated search: coordinate ascent over all 9 thresholds

Cyclic hill-climbing: one full pass grid-searches each threshold in turn, holding the other 8 at
their current best value; passes repeat until a full pass makes no improvement (or `max_passes`
is hit). **Objective**: macro-F1 (per-class harmonic mean of recall/precision, averaged across
classes with RF reference pixels) -- an earlier mean-recall-only objective converged to a
degenerate result (see `notes.md` for why); the grids below are widened past where that run's
optimum landed.

This is a local optimum, not a global one -- treat the result as a candidate, not a final answer:
inspect its confusion matrix below, and it still goes through the mandatory final EE-confirmation
step regardless.


In [ ]:
import time

PARAM_GRIDS = {
    "crops_min": np.round(np.arange(0.10, 0.51, 0.02), 2),
    "built_min": np.round(np.arange(0.05, 0.61, 0.02), 2),
    "water_wetland_min": np.round(np.arange(0.05, 0.61, 0.02), 2),
    "bare_min": np.round(np.arange(0.02, 0.56, 0.02), 2),
    "woody_min": np.round(np.arange(0.05, 0.61, 0.02), 2),
    "grass_min": np.round(np.arange(0.00, 0.61, 0.02), 2),
    "woody_grass_margin": np.round(np.arange(0.00, 0.16, 0.01), 2),
    "natural_min": np.round(np.arange(0.05, 0.51, 0.02), 2),
    "top1_min": np.round(np.arange(0.00, 0.46, 0.02), 2),
}


def per_class_f1(result):
    p = result["per_class"]["producers_accuracy"].to_numpy()
    u = result["per_class"]["users_accuracy"].to_numpy()
    with np.errstate(invalid="ignore", divide="ignore"):
        f1 = np.where((p + u) > 0, 2 * p * u / (p + u), np.nan)
    return f1


def balanced_objective(thresholds):
    result = evaluate_thresholds(overrides=thresholds)
    macro_f1 = np.nanmean(per_class_f1(result))
    return macro_f1, result["kappa"], result["overall_accuracy"]


def coordinate_ascent(param_grids, max_passes=6, tol=1e-4):
    current = dict(config.DW_HABITAT_THRESHOLDS)
    best_score, best_kappa, best_acc = balanced_objective(current)
    history = [
        {"pass": 0, "param": "seed", "value": None, "macro_f1": best_score, "kappa": best_kappa, "overall_accuracy": best_acc}
    ]

    for pass_num in range(1, max_passes + 1):
        improved_this_pass = False
        for param, grid in param_grids.items():
            best_value = current[param]
            local_score, local_kappa, local_acc = best_score, best_kappa, best_acc
            for v in grid:
                trial = dict(current)
                trial[param] = float(v)
                score, kappa, acc = balanced_objective(trial)
                if score > local_score + tol:
                    local_score, local_kappa, local_acc = score, kappa, acc
                    best_value = float(v)
            if best_value != current[param]:
                current[param] = best_value
                best_score, best_kappa, best_acc = local_score, local_kappa, local_acc
                improved_this_pass = True
            history.append(
                {"pass": pass_num, "param": param, "value": current[param], "macro_f1": best_score, "kappa": best_kappa, "overall_accuracy": best_acc}
            )
        print(f"Pass {pass_num}: macro_f1={best_score:.4f}  kappa={best_kappa:.4f}  overall_accuracy={best_acc:.4f}")
        if not improved_this_pass:
            print(f"Converged after {pass_num} pass(es) -- no threshold improved further.")
            break

    return current, pd.DataFrame(history)


t0 = time.time()
best_thresholds, search_history = coordinate_ascent(PARAM_GRIDS, max_passes=6)
print(f"\nSearch took {time.time() - t0:.1f}s")

print("\nBest thresholds found vs. current config.DW_HABITAT_THRESHOLDS:")
for k, v in best_thresholds.items():
    at_lo = np.isclose(v, PARAM_GRIDS[k].min())
    at_hi = np.isclose(v, PARAM_GRIDS[k].max())
    flag = "  <-- AT GRID EDGE" if (at_lo or at_hi) else ""
    print(f"  {k:20s}  {v:.3f}   (was {config.DW_HABITAT_THRESHOLDS[k]:.3f}){flag}")


In [ ]:
candidate_result = evaluate_thresholds(overrides=best_thresholds, label="Coordinate-ascent candidate")


### Why Bare/degraded and Water/flooded veg still get zero predictions

The candidate above still doesn't classify a single Bare/degraded or Water/flooded veg pixel, even
though the search was free to lower `bare_min`/`water_wetland_min` as far as the grid allowed. Two
different causes -- diagnosed by looking at the raw band values *at the RF reference pixels
themselves*, not just sweeping the threshold:


In [ ]:
bare_ref_mask = rf_reference == 6
water_ref_mask = rf_reference == 7

bare_v = bands["bare"].filled(np.nan)
crops_v = bands["crops"].filled(np.nan)
built_v = bands["built"].filled(np.nan)
water_v = bands["water_wetland_prob"].filled(np.nan)

print(f"RF 'bareground' reference pixels: {bare_ref_mask.sum():,}")
print(f"  fraction where bare > crops AND bare > built (the rule's dominance check): "
      f"{((bare_v > crops_v) & (bare_v > built_v))[bare_ref_mask].mean():.1%}")
print(f"  bare value at those pixels: mean={np.nanmean(bare_v[bare_ref_mask]):.3f}  "
      f"median={np.nanmedian(bare_v[bare_ref_mask]):.3f}  max={np.nanmax(bare_v[bare_ref_mask]):.3f}")
print(f"  crops value at those pixels: mean={np.nanmean(crops_v[bare_ref_mask]):.3f}  (often exceeds bare itself)")
print()
print(f"RF 'water' reference pixels: {water_ref_mask.sum():,}")
print(f"  water_wetland_prob at those pixels: mean={np.nanmean(water_v[water_ref_mask]):.3f}  "
      f"median={np.nanmedian(water_v[water_ref_mask]):.3f}  max={np.nanmax(water_v[water_ref_mask]):.3f}")
print(f"  fraction >= 0.10: {(water_v[water_ref_mask] >= 0.10).mean():.1%}   "
      f"fraction >= current threshold (0.35): {(water_v[water_ref_mask] >= 0.35).mean():.1%}")


**Neither class is fixable by threshold alone -- see `notes.md`/`config.py` for the full
diagnosis.** Bare/degraded: true bare pixels average `bare`≈0.08 and `crops`/`built` outscore it
87% of the time, so the rule's dominance check fails structurally regardless of `bare_min` -- a
`classify_habitat()` rule-logic issue, not a threshold one. Water/flooded veg: true water pixels
average `water_wetland_prob`≈0.14 against the 0.35 threshold, but water is so rare here (2,045
reference pixels) that lowering the threshold trades a small recall gain for larger precision
losses elsewhere -- 0.35 is empirically macro-F1-optimal, confirmed by the refinement sweep below.


In [ ]:
water_refine_rows = []
for v in [0.35, 0.25, 0.20, 0.15, 0.12, 0.10, 0.08, 0.06]:
    trial = {**best_thresholds, "water_wetland_min": v}
    result = evaluate_thresholds(overrides=trial)
    macro_f1 = np.nanmean(per_class_f1(result))
    water_row = result["per_class"].loc["Water/flooded veg"]
    water_refine_rows.append(
        {
            "water_wetland_min": v,
            "macro_f1": macro_f1,
            "kappa": result["kappa"],
            "overall_accuracy": result["overall_accuracy"],
            "water_producers_accuracy": water_row["producers_accuracy"],
            "water_users_accuracy": water_row["users_accuracy"],
        }
    )
water_refine_df = pd.DataFrame(water_refine_rows)
water_refine_df


In [ ]:
best_water_row = water_refine_df.loc[water_refine_df["macro_f1"].idxmax()]
final_thresholds = {**best_thresholds, "water_wetland_min": float(best_water_row["water_wetland_min"])}

print("Final candidate thresholds (coordinate-ascent result + refined water_wetland_min):")
for k, v in final_thresholds.items():
    print(f"  {k:20s}  {v:.3f}   (was {config.DW_HABITAT_THRESHOLDS[k]:.3f})")

final_result = evaluate_thresholds(overrides=final_thresholds, label="Final candidate")


### Temporal-mismatch diagnostic: 2025-only composite vs. the 2022-2025 comparison above

The RF ground truth is 2025-only imagery, but every comparison above uses the 2022-2025 four-year
composite used throughout the rest of the pipeline -- this section checks whether that mismatch is
inflating the apparent miscalibration by re-running the same comparison against a parallel
single-year 2025 composite (diagnostic only; see `notes.md` for the conclusion). If
confusion for the time-sensitive classes (Cropland, Bare/degraded, Water/flooded veg) drops
meaningfully here, some of the disagreement above is a temporal artifact rather than a threshold
problem.

**Requires** `calibration_DW_connectivity_inputs_annual_2025_project.tif` and
`calibration_DW_raw_crops_built_bare_annual_2025_project.tif` in `outputs/rasters/dynamic_world/`.
Note `config.DW_MIN_OBS_ANNUAL` applies the same floor regardless of window length, so the
single year will mask out more pixels than the 4-year window -- the scored/unscoreable counts
below quantify that coverage cost directly.


In [ ]:
DW_STACK_2025_PATH = config.DW_RASTER_INPUT_DIR / "calibration_DW_connectivity_inputs_annual_2025_project.tif"
DW_RAW_2025_PATH = config.DW_RASTER_INPUT_DIR / "calibration_DW_raw_crops_built_bare_annual_2025_project.tif"

dw_stack_2025, dw_profile_2025 = read_named_bands(DW_STACK_2025_PATH)
dw_raw_2025, dw_raw_profile_2025 = read_named_bands(DW_RAW_2025_PATH)

assert dw_profile_2025["width"] == dw_profile["width"] and dw_profile_2025["height"] == dw_profile["height"], (
    "2025-only stack is not grid-identical to the 2022-2025 stack -- re-check the export."
)

bands_2025 = {**dw_stack_2025, **dw_raw_2025}
print("Loaded 2025-only bands:", sorted(bands_2025.keys()))


In [ ]:
def evaluate_thresholds_2025(overrides=None, min_obs=None, label=""):
    """Same as evaluate_thresholds(), but classifies bands_2025 instead of the 2022-2025 bands."""
    dw_pred = classify_habitat_np(bands_2025, thresholds=overrides, min_obs=min_obs)
    result = compare_to_reference(dw_pred, rf_reference)
    if label:
        print_comparison(result, label=label)
    return result


baseline_result_2025 = evaluate_thresholds_2025(label="2025-only composite, unmodified config.DW_HABITAT_THRESHOLDS")
print()
final_result_2025 = evaluate_thresholds_2025(overrides=final_thresholds, label="2025-only composite, final_thresholds")


In [ ]:
comparison_rows = []
for label, result in [
    ("2022-2025, unmodified thresholds", baseline_result),
    ("2022-2025, final_thresholds", final_result),
    ("2025-only, unmodified thresholds", baseline_result_2025),
    ("2025-only, final_thresholds", final_result_2025),
]:
    macro_f1 = np.nanmean(per_class_f1(result))
    row = {
        "comparison": label,
        "overall_accuracy": result["overall_accuracy"],
        "kappa": result["kappa"],
        "macro_f1": macro_f1,
        "n_scored_pixels": result["n_scored_pixels"],
        "n_unscoreable_pixels": result["n_unscoreable_pixels"],
    }
    for cls in ["Cropland", "Bare/degraded", "Water/flooded veg"]:
        row[f"{cls}_producers_accuracy"] = result["per_class"].loc[cls, "producers_accuracy"]
    comparison_rows.append(row)

pd.DataFrame(comparison_rows).set_index("comparison")


## Final EE-confirmation step (required, not optional)

Once thresholds are chosen here:

1. Update `config.DW_HABITAT_THRESHOLDS` with the chosen values.
2. Re-run `historical_change_detection.ipynb`'s real `classify_habitat()` for the current-period
   composite only (one-time EE cost) and re-export/download that one raster, replacing
   `DW_class_current_2022_2025_project.tif`.
3. Reclassify locally with the same chosen thresholds (`classify_habitat_np(bands, thresholds=...)`)
   and compare pixel-by-pixel against the freshly downloaded EE raster -- expect near-100%
   agreement; investigate any disagreement before trusting the local reimplementation further.
4. Record the chosen thresholds + confirmation agreement % in `notes.md` as a dated "round 3"
   calibration note, following the existing "round 2 recalibration (2026-07-06)" precedent in
   `config.py`.


In [ ]:
EE_CANDIDATE_PATH = config.DW_RASTER_INPUT_DIR / "calibration_DW_class_current_2022_2025_CANDIDATE_thresholds_project.tif"

with rasterio.open(EE_CANDIDATE_PATH) as src:
    ee_candidate_profile = src.profile
    ee_candidate = src.read(1, masked=True).filled(0).astype("uint8")

assert ee_candidate_profile["width"] == dw_profile["width"] and ee_candidate_profile["height"] == dw_profile["height"], (
    "EE candidate-threshold export is not grid-identical to the local stack -- re-check the export."
)

local_candidate = classify_habitat_np(bands, thresholds=final_thresholds)

agreement = (local_candidate == ee_candidate).mean()
print(f"Pixel agreement between local reimplementation and the real EE classify_habitat() "
      f"(both using final_thresholds): {agreement:.4%}")

disagreement = local_candidate != ee_candidate
print(f"Disagreeing pixels: {disagreement.sum():,} / {disagreement.size:,}")

if disagreement.any():
    mismatch_pairs = pd.DataFrame(
        {"local": local_candidate[disagreement], "ee": ee_candidate[disagreement]}
    ).value_counts().rename("count").reset_index()
    mismatch_pairs["local"] = mismatch_pairs["local"].map(lambda v: config.DW_HABITAT_CLASS_LABELS.get(v, "nodata(0)"))
    mismatch_pairs["ee"] = mismatch_pairs["ee"].map(lambda v: config.DW_HABITAT_CLASS_LABELS.get(v, "nodata(0)"))
    print("\nMost common (local, EE) disagreement pairs:")
    print(mismatch_pairs.head(15).to_string(index=False))
